## **ReAct: Build Reasoning and Acting AI Agents with LangGraph**

#### 1. Web Search Tool
### Tavily Search API Key Setup

In [1]:
import warnings 
warnings.filterwarnings('ignore')

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool
import os
import json

#tavily_api_key = os.environ.get("TAVILY_API_KEY")
from dotenv import load_dotenv
load_dotenv()

# Initialize the Tavily search tool
search = TavilySearchResults()

@tool
def search_tool(query: str):
    """
    Search the web for information using Tavily API.

    :param query: The search query string
    :return: Search results related to the query
    """
    return search.invoke(query)

/var/folders/r7/5lkygyb92bqb1_3pfblc7wym0000gn/T/ipykernel_1711/2304387661.py:14: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search = TavilySearchResults()


### Testing the Search Tool

In [2]:
search_tool.invoke("What's the weather like in Tokyo today?")

[{'title': 'Tokyo Weather Conditions: Temperature | 30 Days Forecast',
  'url': 'https://www.aqi.in/weather/us/japan/tokyo/tokyo',
  'content': "From 04:30 AM 10 September 2026, Tokyo's 10-day forecast shows this trend:\nToday (Sep. 10) : Temp. 21°C, Hum. 81% and Patchy rain nearby condition.\nFriday (Sep. 11) : Temp. 20°C, Hum. 85% and Rain Shower condition.\nSaturday (Sep. 12) : Temp. 21°C, Hum. 85% and Patchy rain nearby condition.\nSunday (Sep. 13) : Temp. 25°C, Hum. 76% and Overcast condition.\nMonday (Sep. 14) : Temp. 27°C, Hum. 70% and Overcast condition.\nTuesday (Sep. 15) : Temp. 25°C, Hum. 80% and Patchy rain nearby condition.\nWednesday (Sep. 16) : Temp. 25°C, Hum. 80% and Patchy rain nearby condition.\nThursday (Sep. 17) : Temp. 27°C, Hum. 74% and Patchy rain nearby condition.\nFriday (Sep. 18) : Temp. 25°C, Hum. 71% and Rain Showers condition.\nSaturday (Sep. 19) : Temp. 24°C, Hum. 71% and Rain Showers condition. [...] 2. What are the current weather parameters in Tokyo to

#### 2. Clothing Recommendation Tool

In [3]:
@tool
def recommend_clothing(weather: str) -> str:
    """
    Returns a clothing recommendation based on the provided weather description.

    This function examines the input string for specific keywords or temperature indicators 
    (e.g., "snow", "freezing", "rain", "85°F") to suggest appropriate attire. It handles 
    common weather conditions like snow, rain, heat, and cold by providing simple and practical 
    clothing advice.

    :param weather: A brief description of the weather (e.g., "Overcast, 64.9°F")
    :return: A string with clothing recommendations suitable for the weather
    """
    weather = weather.lower()
    if "snow" in weather or "freezing" in weather:
        return "Wear a heavy coat, gloves, and boots."
    elif "rain" in weather or "wet" in weather:
        return "Bring a raincoat and waterproof shoes."
    elif "hot" in weather or "85" in weather:
        return "T-shirt, shorts, and sunscreen recommended."
    elif "cold" in weather or "50" in weather:
        return "Wear a warm jacket or sweater."
    else:
        return "A light jacket should be fine."

#### Creating the Tool Registry

In [4]:
tools=[search_tool,recommend_clothing]

tools_by_name={ tool.name:tool for tool in tools}

## Setting Up the Language Model

### Initializing the AI Model

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

openai_api_key = os.getenv("OPENAI_KEY")

model = ChatOpenAI(model="gpt-4o-mini", api_key=openai_api_key)

### Creating the System Prompt

In [6]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage,SystemMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a helpful AI assistant that thinks step-by-step and uses tools when needed.

When responding to queries:
1. First, think about what information you need
2. Use available tools if you need current data or specific capabilities  
3. Provide clear, helpful responses based on your reasoning and any tool results

Always explain your thinking process to help users understand your approach.
"""),
    MessagesPlaceholder(variable_name="scratch_pad")
])

### Binding Tools to the Model

In [7]:
model_react=chat_prompt|model.bind_tools(tools)

## Setting Agent State

The agent must maintain context across multiple reasoning and acting steps

In [8]:
from typing import (Annotated,Sequence,TypedDict)
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    """The state of the agent."""

    # add_messages is a reducer
    # Link for more details: https://langchain-ai.github.io/langgraph/concepts/low_level/#reducers
    messages: Annotated[Sequence[BaseMessage], add_messages]

**Key Concepts:**
- **State**: Contains the conversation history and context.
- **Reducer**: `add_messages` automatically handles adding new messages to the conversation.
- **Type Safety**: TypedDict ensures our state structure is well-defined.

### Demonstrating State Management

In [9]:
state: AgentState = {"messages": []}

# append a message using the reducer properly
state["messages"] = add_messages(state["messages"], [HumanMessage(content="Hi")])
print("After greeting:", state["messages"])

# add another message (e.g. a question)
state["messages"] = add_messages(state["messages"], [HumanMessage(content="Weather in NYC?")])
print("After question:", state)

After greeting: [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='6bf6a99a-58f5-4d67-bd12-45404f17680d')]
After question: {'messages': [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='6bf6a99a-58f5-4d67-bd12-45404f17680d'), HumanMessage(content='Weather in NYC?', additional_kwargs={}, response_metadata={}, id='d46f99e4-5671-49ae-b6b8-6732f613c6dc')]}


This demonstrates how the state accumulates context over the conversation.

## Manual ReAct Execution

Manually steping through a ReAct cycle:

### Step 1: Initial Query Processing

In [10]:
dummy_state: AgentState = {
    # The user asks a complex question requiring current data.
    "messages": [HumanMessage( "What's the weather like in Zurich, and what should I wear based on the temperature?")]}

# The model analyzes the query and realizes it needs to search for weather information.
# The model generates a tool call for the search.
response = model_react.invoke({"scratch_pad":dummy_state["messages"]})

dummy_state["messages"]=add_messages(dummy_state["messages"],[response])

### Step 2: Tool Execution

In [11]:
# Extract the tool call from the model's response.
tool_call = response.tool_calls[-1]
print("Tool call:", tool_call)

# Execute the tool using the specified arguments.
tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
print("Tool result preview:", tool_result[0]['title'])

# Create a ToolMessage containing the results.
tool_message = ToolMessage(
    content=json.dumps(tool_result),
    name=tool_call["name"],
    tool_call_id=tool_call["id"]
)

# Add the tool result to the conversation state.
dummy_state["messages"] = add_messages(dummy_state["messages"], [tool_message])

Tool call: {'name': 'search_tool', 'args': {'query': 'current weather Zurich'}, 'id': 'call_deekKhZKaZV3NLbIAUbSwknh', 'type': 'tool_call'}
Tool result preview: Zürich (Kreis 2), Zurich Weather Forecast | WindBorne Systems


### Step 3: Processing Results and Next Action

In [12]:
# The model processes the search results.
response = model_react.invoke({"scratch_pad": dummy_state["messages"]})
dummy_state['messages'] = add_messages(dummy_state['messages'], [response])

# check if the model wants to use another tool
if response.tool_calls:
    tool_call = response.tool_calls[0]
    tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
    tool_message = ToolMessage(
        content=json.dumps(tool_result),
        name=tool_call["name"],
        tool_call_id=tool_call["id"]
    )
    dummy_state['messages'] = add_messages(dummy_state['messages'], [tool_message])

### Step 4: Final Response Generation

In [13]:
response = model_react.invoke({"scratch_pad": dummy_state["messages"]})
print("Final response generated:", response.content is not None)
print("More tools needed:", bool(response.tool_calls))

Final response generated: True
More tools needed: False


## Automating ReAct with Graphs

There is no reason to run manually the execute application since LangGraph automates this process with a state machine that handles the reasoning loop automatically.


#### Tool Execution Node


In [14]:
def tool_node(state: AgentState):
    """Execute all tool calls from the last message in the state."""
    outputs = []
    for tool_call in state["messages"][-1].tool_calls:
        tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
        outputs.append(
            ToolMessage(
                content=json.dumps(tool_result),
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )
    return {"messages": outputs}

#### Model Invocation Node

In [15]:
def call_model(state: AgentState):
    """Invoke the model with the current conversation state."""
    response = model_react.invoke({"scratch_pad": state["messages"]})
    return {"messages": [response]}

#### Decision Logic

In [16]:
def should_continue(state: AgentState):
    """Determine whether to continue with tool use or end the conversation."""
    messages = state["messages"]
    last_message = messages[-1]
    # If there is no function call, then we finish
    if not last_message.tool_calls:
        return "end"
    # Otherwise if there is, we continue
    else:
        return "continue"

### Constructing the State Graph

In [17]:
from langgraph.graph import StateGraph, END

# Define a new graph
workflow = StateGraph(AgentState)

# Define the two nodes we will cycle between
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

# Add edges between nodes
workflow.add_edge("tools", "agent")  # After tools, always go back to agent

# Add conditional logic
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {
        "continue": "tools",  # If tools needed, go to tools node
        "end": END,          # If done, end the conversation
    },
)

# Set entry point
workflow.set_entry_point("agent")

# Compile the graph
graph = workflow.compile()